# 04 — Resource Utilisation Analysis

This notebook addresses **RQ3**: *How does CPU and memory consumption compare
between the .NET MediatR stack and the Go goroutine model under equivalent load?*

We derive a key efficiency metric — **requests per CPU percent** — that normalises
throughput by resource cost, giving a fair single-number comparison independent
of raw RPS differences.

## Imports and setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..') / 'scripts'))
from utils import SERVICES, SERVICE_LABELS, SERVICE_COLORS, load_csv, describe_series, set_plot_style
from statistical_tests import _compute_test_result
from generate_charts import fig04_resource_utilisation

set_plot_style()
%matplotlib inline

## Load CPU and memory data

CPU is expressed as a percentage of one core (Prometheus
`process_cpu_seconds_total` rate × 100).  Memory is the in-use heap bytes
converted to MB for readability.

In [ ]:
cpu_dn  = load_csv('cpu_dotnet.csv')['cpu_percent']
cpu_go  = load_csv('cpu_go.csv')['cpu_percent']
mem_dn  = load_csv('memory_heap_dotnet.csv')['bytes'] / (1024 ** 2)
mem_go  = load_csv('memory_heap_go.csv')['bytes']     / (1024 ** 2)

print(f'.NET CPU  samples: {len(cpu_dn)} | Go CPU  samples: {len(cpu_go)}')
print(f'.NET Heap samples: {len(mem_dn)} | Go Heap samples: {len(mem_go)}')

## Descriptive statistics: CPU and memory

Peak CPU and peak heap highlight worst-case resource spikes, while the mean
gives the steady-state cost of running each service.

In [ ]:
rows = []
for label, s, unit in [
    ('.NET CPU',  cpu_dn, '%'), ('Go CPU',  cpu_go, '%'),
    ('.NET Heap', mem_dn, 'MB'), ('Go Heap', mem_go, 'MB'),
]:
    d = describe_series(s)
    rows.append([label, f"{d['mean']:.2f}", f"{d['std']:.2f}",
                 f"{d['p95']:.2f}", f"{d['max']:.2f}", unit])

headers = ['Series', 'Mean', 'Std', 'P95', 'Peak', 'Unit']
pd.DataFrame(rows, columns=headers)

## Mann-Whitney U — CPU usage

In [ ]:
r_cpu = _compute_test_result(
    cpu_dn.dropna().to_numpy(), cpu_go.dropna().to_numpy(),
    'CPU Usage', '%', n_bootstrap=10_000,
)
print(f'CPU -- p-value: {r_cpu.p_value:.6f}  winner: {r_cpu.winner.upper()}  effect: {r_cpu.effect_label}')
print(f'  .NET mean: {r_cpu.dotnet_mean:.2f}%  |  Go mean: {r_cpu.go_mean:.2f}%')
print(f'  95% CI on diff: [{r_cpu.ci_low:+.2f}, {r_cpu.ci_high:+.2f}]%')

## Mann-Whitney U — memory usage

In [ ]:
r_mem = _compute_test_result(
    mem_dn.dropna().to_numpy(), mem_go.dropna().to_numpy(),
    'Heap Memory', 'MB', n_bootstrap=10_000,
)
print(f'Memory -- p-value: {r_mem.p_value:.6f}  winner: {r_mem.winner.upper()}  effect: {r_mem.effect_label}')
print(f'  .NET mean: {r_mem.dotnet_mean:.2f} MB  |  Go mean: {r_mem.go_mean:.2f} MB')
print(f'  95% CI on diff: [{r_mem.ci_low:+.2f}, {r_mem.ci_high:+.2f}] MB')

## Figure 4: resource utilisation plots

In [ ]:
fig04_resource_utilisation()

## Efficiency metric: requests per CPU percent

Dividing mean RPS by mean CPU% gives us a hardware-normalised throughput figure.
A higher value means the service does more work for the same CPU budget.

In [ ]:
try:
    rps_dn = load_csv('throughput_dotnet.csv')['rps']
    rps_go = load_csv('throughput_go.csv')['rps']
except FileNotFoundError as e:
    print(f'Throughput data missing: {e}')
else:
    eff_dn = rps_dn.mean() / cpu_dn.mean() if cpu_dn.mean() > 0 else float('nan')
    eff_go = rps_go.mean() / cpu_go.mean() if cpu_go.mean() > 0 else float('nan')
    print(f'.NET efficiency : {eff_dn:.2f} req/s per CPU%')
    print(f'Go   efficiency : {eff_go:.2f} req/s per CPU%')
    if eff_go > 0:
        print(f'Go is {(eff_go / eff_dn - 1) * 100:+.1f}% more efficient per CPU percent')

## Interpretation

**Fill in after running with real data.**

Template:

> The **Go service used X% less CPU on average** (p=XXXX, Cliff's delta=X.XX,
> Y effect) and **X MB less heap memory** (p=XXXX).  The Go efficiency metric of
> A req/s per CPU% compares favourably to .NET's B req/s per CPU%, a C% improvement.
>
> The lower .NET heap figure partially reflects the CLR's generational GC
> releasing memory between GC cycles rather than keeping it mapped; the Go
> runtime retains more pre-allocated goroutine stacks.
>
> **Dissertation answer (RQ3):** Go consumes fewer CPU resources per request,
> making it the better choice for resource-constrained deployments.